# Neural Network

### By Jorge Almonacid

Problem definition: given a set of data, apply a neural network with forwarding and backpropagation, with adam optimizer and MSE loss function 

In [17]:
import numpy as np

First we define our loss function, MSE: 


$L = \frac{1}{2}(y_p-y_r)^2$


And also define its dereivative to apply the optimizer:

$L' = (y_p - y_r)$


Where $y_p$ are the predictions and $y_r$ the real values

In [18]:
# Loss
def mse(y_pred, y_real):
    return (1/2)*(y_pred-y_real)**2

def mse_gradient(y_pred, y_real):
    return (y_pred-y_real)

mse.diff = mse_gradient

Now we define the most common activation functions used in neural networks and their derivatives:

- Sigmoid (used for binary classification networks): $\frac{1}{1+e^{-x}}$
- Linear (used for regression problems): g(x) = x.
- ReLU (used as a hidden layer activation function): g(x) = $\begin{cases}
    x & \text{if } x > 0 \\
    0 & \text{if } x < 0
\end{cases}$


Additionally we define a function to apply to the data the activation function

In [19]:
# Activation functions
def sigmoid(x:np.ndarray):
    return 1/(1+np.exp(-x))

def sig_diff(x:np.ndarray):
    return sigmoid(x)*(1-sigmoid(x))

sigmoid.diff = sig_diff

def linear(x:np.ndarray):
    return x

def linear_diff(x:np.ndarray):
    return np.ones_like(x)
    
linear.diff = linear_diff
    
def relu(x:list):
    return np.maximum(0, x)

def relu_diff(x:np.ndarray | list):
    return np.where(x>0, 1.0, 0.0)

relu.diff = relu_diff

def set_activation_function(act_function):
    def apply_function(weights,values):
        x = np.dot(weights, values)
        return act_function(x)
    return apply_function

Finally we define our optimizer, ADAM in our case. ADAM ecuations take into account how fast the gradient was moving the last step to find a global minimum for the loss function. Equations:


$m_0 = 0, v_0 = 0$


$m_{t+1} \longleftarrow \beta_1 m_t + (1-\beta_1) \nabla_{\theta} \mathcal{L}(\theta)$


$v_{t+1} \longleftarrow \beta_2 v_t + (1-\beta_2)\nabla_{\theta} \mathcal{L}^2$


$\theta_{j} \longleftarrow \theta_j - \frac{\epsilon}{\sqrt{v_{t+1}}+1e^{-5}} m_{t+1}$

In [20]:
# # ADAM optimizer

m_0 = 0
v_0 = 0

def v_m(vel, m, grad, theta, optimizer="adam", beta1=0.9, beta2=0.999, learn_rate= 0.01, epsilon= 1e-8):
    
    m_1, v_1= m,vel
    if optimizer == "adam":
        m_1 = beta1*m + (1-beta1)*grad
        v_1 = beta2*vel + (1-beta2)*grad**2
            
        theta_j = theta - (learn_rate*m_1)/(np.sqrt(v_1) + epsilon)
    
    else: 
        theta_j = optimizer(m_1, v_1)
        

    return m_1, v_1, theta_j

Now we define the logic behind the layers, we will be using only fully connected layers (dense layer). First we define a memory that will help us later when applying the forward pass and backpropagation. 

In [21]:
# Definimos la capa densa que usaremos en el modelo

def dense_layer(neuronas, activation= relu):
    memoria = {
        "w": None, #pesos
        "b": None, #bias
        "m_w": None, #momentos
        "v_w": None, #velocidad
        "m_b": None, #momento bias
        "m_v": None, #velocidad bias
        "act_func": activation, # Función de activación
        "input": None
    } 
    
    def aplicar_capa(data, training = True):
        if memoria["w"] is None:
            n_inputs = data.shape[1]
            memoria["w"] = np.random.rand(n_inputs,neuronas)
            memoria["b"] = np.zeros((1,neuronas))
            
            # Momentos y velocidades
            memoria["m_w"] = np.zeros_like(memoria["w"]) 
            memoria["m_b"] = np.zeros_like(memoria["b"]) 
            memoria["v_w"] = np.zeros_like(memoria["w"]) 
            memoria["v_b"] = np.zeros_like(memoria["b"]) 

        if training:
            memoria["input"] = data
        
        z = data @ memoria["w"] + memoria["b"]
    
        return activation(z), memoria
    
    return aplicar_capa

This next function, given the memory and the gradient passed from the last layer, calculates the new gradient and performs the backpropagation.


Backpropagation: 


$z_i^j = X_{i-1}^j·W_i^j+b^j$, where $X_{i-1}^j$ is the data matrix passed from the previous layer, $W_i^j$ is the weight matrix and $b^j$ is the bias vector.

$\delta_i = e_i · \frac{\partial a}{\partial X}$, with $e_i = \frac{\partial L}{\partial a_{n-i+1}}$


$\nabla W_{n-i+1} = \delta_i · X_{n-i+1}$


In [22]:
def train_layer(grad_output, memoria, learn_rate=0.01):
    """
    Realiza el Backpropagation completo para una capa dada su memoria.
    """
    # Datos que la capa generó en el forwardpass
    W_viejo = memoria["w"]
    b_viejo = memoria["b"]
    X = memoria["input"]       
    act = memoria["act_func"]    
    
    # Datos antiguos
    z = X @ W_viejo + b_viejo
    
    # Backpropagation
    delta = grad_output * act.diff(z) 
    error_anterior = delta @ W_viejo.T
    
    # Calcular los gradientes
    grad_w = X.T @ delta
    grad_b = np.sum(delta, axis=0, keepdims=True)
    
    # Actualización de los pesos
    memoria["m_w"], memoria["v_w"], memoria["w"] = v_m(
        memoria["v_w"], memoria["m_w"], grad_w, memoria["w"], 
        optimizer="adam", learn_rate=learn_rate
    )
    
    memoria["m_b"], memoria["v_b"], memoria["b"] = v_m(
        memoria["v_b"], memoria["m_b"], grad_b, memoria["b"], 
        optimizer="adam", learn_rate=learn_rate
    )

    return error_anterior

Lastly we create the data we want to model and apply the forward pass and backpropagation to train the NN, to then check with a validation set how is our model's accuracy and predict a value with a test value.

In [23]:
def generate_dataset(n_samples):
    m1 = np.random.uniform(1e20, 1e24, n_samples)
    m2 = np.random.uniform(1e3, 1e5, n_samples)
    r  = np.random.uniform(1e4, 1e6, n_samples)
    h  = np.random.uniform(-r/2, 2000, n_samples) 
    
    G = 6.674e-11
    R_total = r + h
    F = []
    
    for i in range(n_samples):
        noise = np.random.uniform(0.99, 1.01) 
        if h[i] >= 0: 
            val = (G * m1[i] * m2[i] / (R_total[i]**2)) * noise
        else: 
            val = (G * m1[i] * m2[i] * R_total[i] / (r[i]**3)) * noise
        F.append(val)
        
    return np.column_stack((m1, m2, r, h, np.array(F)))

train_set = generate_dataset(10000)
val_set   = generate_dataset(2000)

# Preprocesamiento
m1_tr, m2_tr, r_tr, h_tr, F_tr = train_set[:,0], train_set[:,1], train_set[:,2], train_set[:,3], train_set[:,4]
m1_va, m2_va, r_va, h_va, F_va = val_set[:,0],   val_set[:,1],   val_set[:,2],   val_set[:,3],   val_set[:,4]

# Alturas no nulas para no dividir por 0
R_tr = np.maximum(r_tr + h_tr, 1e-9)
R_va = np.maximum(r_va + h_va, 1e-9)

# Datos a espacio logarítmico
z_tr = np.where(h_tr >= 0, -2*np.log(R_tr), np.log(R_tr) - 3*np.log(r_tr))
z_va = np.where(h_va >= 0, -2*np.log(R_va), np.log(R_va) - 3*np.log(r_va))

# Inputs
x_train = np.stack([np.log(m1_tr), np.log(m2_tr), z_tr], axis=1)
x_val   = np.stack([np.log(m1_va), np.log(m2_va), z_va], axis=1)

# Función a predecir
y_train = np.log(F_tr.reshape(-1,1))
y_val   = np.log(F_va.reshape(-1,1))

# Hiperparámetros
epochs = 20000
lr = 0.001 

print(f"Inicio del entrenamiento Deep Learning.")
print(f"Input Shape: {x_train.shape} | Output Shape: {y_train.shape}")
print("-" * 40)

# Arquitectura 3 -> 8 -> 8 -> 1
# Hidden Layers
capa1 = dense_layer(neuronas=8, activation=relu)
capa2 = dense_layer(neuronas=16, activation=relu)
# capa6 = dense_layer(neuronas=16, activation=relu)
capa3 = dense_layer(neuronas=8, activation=relu)
# Output layer
capa4 = dense_layer(neuronas=1, activation=linear)

# Entrenamiento de la red
for i in range(epochs):
    
    # Forward pass
    # Pasamos la señal capa por capa y guardamos las memorias
    out1, mem1 = capa1(x_train)    # Entrada -> Oculta 1
    out2, mem2 = capa2(out1)       # Oculta 1 -> Oculta 2
    # out5, mem5 = capa5(out2)       # Oculta 1 -> Oculta 2
    # out6, mem6 = capa6(out2)       # Oculta 1 -> Oculta 2
    out3, mem3 = capa3(out2)     # Oculta 2 -> Oculta 3
    y_pred, mem4 = capa4(out3)     # Oculta 4 -> Salida
     
    # Cáluclo de la pérdida (MSE)
    loss = np.mean(mse(y_pred, y_train))
    
    # Backpropagation
    # Primer gradiente
    grad = mse_gradient(y_pred, y_train)
    
    # 2. Retropropagación a la siguiente capa
    # Capa 4 (Salida) -> Devuelve gradiente para Capa 3
    grad = train_layer(grad, mem4, learn_rate=lr)
    
    grad = train_layer(grad, mem3, learn_rate=lr)
    # Capa 3 (Oculta) -> Devuelve gradiente para Capa 2
    # grad = train_layer(grad, mem6, learn_rate=lr)
    # grad = train_layer(grad, mem5, learn_rate=lr)
    # Capa 2 (Oculta) -> Devuelve gradiente para Capa 1
    grad = train_layer(grad, mem2, learn_rate=lr)
    
    
    # Capa 1 (Entrada) -> Devuelve gradiente para input
    grad = train_layer(grad, mem1, learn_rate=lr)
    
    # Cálculo de la red sobre el set de validación (forward pass solo) para calcular métricas
    if i % 10 == 0:
        val_out1, _ = capa1(x_val, training=False)
        val_out2, _ = capa2(val_out1, training=False)
        # val_out5, _ = capa5(val_out2, training=False)
        # val_out6, _ = capa6(val_out2, training=False)
        val_out3, _ = capa3(val_out2, training=False)
        val_pred, _ = capa4(val_out3, training=False)
        
        val_loss = np.mean(mse(val_pred, y_val))
        print(f"Epoch {i:4d} | Train Loss: {loss:.6f} | Val Loss: {val_loss:.6f}")

print("-" * 40)
print("Entrenamiento completado.")

# Forward pass sobre datos de test

idx = 0
real_val = y_val[idx][0]
input_sample = x_val[idx:idx+1] # Tomamos una muestra

# Predecimos
p1, _ = capa1(input_sample, training=False)
p2, _ = capa2(p1, training=False)
# p5, _ = capa5(p2, training=False)
# p6, _ = capa6(p2, training=False)
p3, _ = capa3(p2, training=False)
pred_val, _ = capa4(p3, training=False)
pred_val = pred_val[0][0]

print(f"\nPrueba ciega (Muestra {idx}):")
print(f"Valor Real (Log F):      {real_val:.4f}")
print(f"Predicción Red (Log F):  {pred_val:.4f}")
print(f"Error Absoluto:          {abs(real_val - pred_val):.4f}")

# Revertimos logaritmo para ver Fuerza en Newtons
F_real_N = np.exp(real_val)
F_pred_N = np.exp(pred_val)
print(f"\nFuerza Real:      {F_real_N:.2e} N")
print(f"Fuerza Predicha:  {F_pred_N:.2e} N")

Inicio del entrenamiento Deep Learning.
Input Shape: (10000, 3) | Output Shape: (10000, 1)
----------------------------------------
Epoch    0 | Train Loss: 4000328.375620 | Val Loss: 3743468.691678
Epoch   10 | Train Loss: 1205386.775629 | Val Loss: 1053261.330480
Epoch   20 | Train Loss: 392109.787581 | Val Loss: 361895.787214
Epoch   30 | Train Loss: 203460.828023 | Val Loss: 193546.785889
Epoch   40 | Train Loss: 133243.364754 | Val Loss: 128912.231287
Epoch   50 | Train Loss: 99438.646131 | Val Loss: 97109.035630
Epoch   60 | Train Loss: 79733.628528 | Val Loss: 78267.481554
Epoch   70 | Train Loss: 66559.486222 | Val Loss: 65531.017905
Epoch   80 | Train Loss: 56967.463685 | Val Loss: 56203.758075
Epoch   90 | Train Loss: 49596.031416 | Val Loss: 48995.125974
Epoch  100 | Train Loss: 43664.590632 | Val Loss: 43176.798933
Epoch  110 | Train Loss: 38770.427212 | Val Loss: 38366.204226
Epoch  120 | Train Loss: 34660.899115 | Val Loss: 34320.786138
Epoch  130 | Train Loss: 31164.0124